In [ ]:
from dataclasses import dataclass
from datetime import time
from geopy.point import Point 
from typing import Self
import re


def convert_to_24_hour_time(time_to_normalize: str) -> time:
    match: re.Match[str] | None = re.match(r"(\d{2}):(\d{2}):(\d{2})", time_to_normalize)
    if not match:
        raise ValueError(f"Invalid time format: {time_to_normalize}")
    
    hour, minute, second = map(int, match.groups())
    
    if hour >= 24:
        hour -= 24
    
    return time(hour, minute, second)

@dataclass 
class Node:
    name: str
    location: Point
     

@dataclass
class CommunicationStep:
    company: str
    line: str
    departure_time: time
    arrival_time: time
    start_stop: Node
    end_stop: Node
    
    @staticmethod
    def from_parsed_csv_line(row: list[str]) -> "CommunicationStep":
        start_stop_point: Point = Point(latitude=row[7], longitude=row[8])
        start_stop = Node(row[5], start_stop_point)
        
        end_stop_point: Point = Point(latitude=row[9], longitude=row[10])
        end_stop = Node(row[6], end_stop_point)
        
        company, line, departure_str, arrival_str = row[1:5]
        
        departure_time: time = convert_to_24_hour_time(departure_str)
        arrival_time: time = convert_to_24_hour_time(arrival_str)
        
        new_communication_step = CommunicationStep(company, line, departure_time, arrival_time, start_stop, end_stop)
        
        return new_communication_step

In [7]:
import pandas as pd
    
df: pd.DataFrame = pd.read_csv("data/connection_graph.csv", dtype={"line": str}, skiprows=900000)

In [8]:
# cleaning data

In [ ]:
from collections import defaultdict

class Graph:
    def __init__(self) -> None:
        self.nodes = {}
        self.edges = defaultdict(list)
        self.adjacency_list = defaultdict(list)
        
        
graph = Graph()

for index, row in df.iterrows():
    step: CommunicationStep = CommunicationStep.from_parsed_csv_line(list(row))
    
    for stop in [step.start_stop, step.end_stop]:
        if stop.name not in graph.nodes:
            graph.nodes[stop.name] = stop
    
    key: tuple[str, str] = (step.start_stop.name, step.end_stop.name)
    if key not in graph.edges:
        graph.edges[key] = []
    graph.edges[key].append(step)

Do każdego z zadań przygotuj raport zawierający opis teoretyczny metody, przykładowe zastosowania, wprowadzone modyfikacje, materiały dodatkowe oraz krótkie opisy bibliotek wykorzystanych przy implementacji. W podsumowaniu raportu dodatkowo opisz napotkane problemy implementacyjne przy wykonywaniu zadania. Raport wyślij prowadzącemu przynajmniej na 24 godziny przed
oddaniem listy.

Wykorzystując udostępniony plik connection_graph.csv zaimplementuj algorytm wyszukiwania najkrótszych połączeń pomiędzy zadanymi przystankami A i B. Jako miarę odległości przyjmij, zależnie od decyzji użytkownika, czas dojazdu z A do B lub liczbę przesiadek koniecznych do wykonania.

Program powinien przyjmować na wejściu wyłącznie 4 zmienne:

- przystanek początkowy A
- przystanek końcowy B
- kryterium optymalizacyjne: wartość t oznacza minimalizację czasu dojazdu, wartość p oznacza minimalizację liczby zmian linii
- czas pojawienia się na przystanku początkowym

Program powinien zwracać na standardowym wyjściu harmonogram przejazdu, wypisując w kolejnych liniach informacje o kolejno wykorzystanych
liniach komunikacyjnych (nazwa linii, czas i przystanek, na którym wsiadamy do danej linii komunikacyjnej oraz czas i przystanek, na którym
kończymy korzystać z danej linii). Na standardowym wyjściu błędów powinien wypisywać wartość funkcji kosztu znalezionego rozwiązania oraz
czas obliczeń liczony od wczytania danych do uzyskania rozwiązania.

Punktacja:

<ol type="a">
  <li>algorytm wyszukiwania najkrótszej ścieżki z A do B za pomocą algorytmu algorytmem Dijkstry w oparciu o kryterium czasu (10 punktów)</li>
  <li>algorytm wyszukiwania najkrótszej ścieżki z A do B za pomocą algorytmu A* w oparciu o kryterium czasu (25 punktów)</li>
  <li>algorytm wyszukiwania najkrótszej ścieżki z A do B za pomocą algorytmu A* w oparciu o kryterium przesiadek (25 punktów)</li>
  <li>modyfikacja algorytmu A* z punktów (b) lub (c), który pozwoli na zmniejszenie wartości funkcji kosztu uzyskanego rozwiązania lub czasu obliczeń (10 punktów)</li>
</ol>

In [ ]:
a = input("podaj przystanek początkowy A: ")
b = input("podaj przystanek końcowy B: ")
optimization_criterium = input("podaj kryterium optymalizacyjne: wartość t oznacza minimalizację czasu dojazdu, wartość p oznacza minimalizację liczby zmian linii")
start_time = input("czas pojawienia się na przystanku początkowym")

In [12]:
print(graph.nodes)
a = "DWORZEC AUTOBUSOWY"
b = "Dyrekcyjna"
optimization_criterium = "t"
start_time = "15:20:00"

def algorithm_a_to_b(a, b, optimization_criterium, start_time):
    start_node = graph.nodes[a]
    end_node = graph.nodes[a]
    
algorithm_a_to_b(a, b, optimization_criterium, start_time)

{'DWORZEC AUTOBUSOWY': Node(name='DWORZEC AUTOBUSOWY', location=Point(51.09726779, 17.03228367, 0.0)), 'Dyrekcyjna': Node(name='Dyrekcyjna', location=Point(51.09430136, 17.03222909, 0.0)), 'PETRUSEWICZA': Node(name='PETRUSEWICZA', location=Point(51.09213757, 17.03100544, 0.0)), 'LEŚNICA': Node(name='LEŚNICA', location=Point(51.14471515, 16.87125685, 0.0)), 'Średzka': Node(name='Średzka', location=Point(51.14535622, 16.86660794, 0.0)), 'Jeleniogórska': Node(name='Jeleniogórska', location=Point(51.14562834, 16.87602438, 0.0)), 'Śnieżna': Node(name='Śnieżna', location=Point(51.14968428, 16.87935804, 0.0)), 'Ciechocińska': Node(name='Ciechocińska', location=Point(51.152764, 16.884055, 0.0)), 'Wojanowska': Node(name='Wojanowska', location=Point(51.15402867, 16.89489676, 0.0)), 'Arachidowa': Node(name='Arachidowa', location=Point(51.15417321, 16.89901728, 0.0)), 'Olbrachtowska': Node(name='Olbrachtowska', location=Point(51.152698, 16.904995, 0.0)), 'Stoszowska': Node(name='Stoszowska', locat